# 68장 · 서비스 함수를 HTTP로 제공하기

과제 안내를 화면에서 요청하려면 요청 주소와 자료 형식이 필요합니다. FastAPI 앱에 POST /ask를 붙이고 TestClient가 실제 앱에 보낸 HTTP 요청을 관찰합니다. 서버 포트를 열지 않아도 경로와 검증을 확인할 수 있습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 fastapi 0.135.0, pydantic 2.14.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 경로에 도착하기 전에 요청 검사하기

Pydantic 모델은 question과 k를 파싱합니다. 질문 앞뒤 공백은 정리하고 공백만 남은 질문은 거절합니다. k에는 1~10 범위의 정수만 받습니다. 아래의 수업용 서비스는 고정 문서 한 건에서 A1 결과를 만듭니다.


### 셀 01 · 제공 셀 · 앱과 입력 모델

요청을 검사할 모델을 먼저 등록합니다.

In [1]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, Field, StrictInt, field_validator
app = FastAPI()
class AskRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    question: str = Field(min_length=1, max_length=1000)
    k: StrictInt = Field(default=2, ge=1, le=10)
    @field_validator("question")
    @classmethod
    def strip_question(cls, value):
        if not value.strip():
            raise ValueError("blank_question")
        return value.strip()


AskRequest가 정의됐습니다. 아직 질문은 전송하지 않았습니다.

### 셀 02 · POST 한 건을 서비스 함수에 연결하기

question은 JSON 본문으로 보내세요. 함수에 건네는 문자열과 응답의 근거를 확인합니다.

이 짧은 경로에서 question은 검색에 들어가고 k는 요청 형식을 검사하는 데 쓰입니다. k를 바꿨는데 문서 개수가 그대로라면, 검사 뒤 검색 함수까지 그 값이 전달되는지 살펴보세요.

In [2]:
def answer(question):
    source = {"id": "course-guide-v1:p2", "page": 1, "source": "가상 과정 운영 안내",
              "text": "과제는 학습 관리 페이지의 과제함에 제출합니다."}
    if "과제" in question:
        return {"status": "answered", "answer": source["text"] + " [course-guide-v1:p2]",
                "sources": [source], "generation_calls": 1}
    return {"status": "no_evidence", "answer": "자료에서 확인할 수 없습니다.", "sources": [], "generation_calls": 0}

@app.post("/ask")
def ask(body: AskRequest):
    return {"run_status": "completed", "result": answer(body.question)}

client = TestClient(app)
response = client.post("/ask", json={"question": "과제는 어디에 내나요?", "k": 2})
print(response.status_code, response.json()["result"]["sources"][0]["id"])

200 course-guide-v1:p2


200 뒤의 course-guide-v1:p2는 서비스가 돌려준 근거입니다. HTTP 상태만으로 이 문서가 답을 뒷받침하는지 판단할 수는 없습니다. 여기서 사용한 응답 외피는 핵심 필드만 담았습니다.

## 형식 오류와 근거 없음 구별하기

입력을 읽을 수 없는 경우와, 입력은 정상이지만 근거를 찾지 못한 경우에 화면이 해야 할 일이 다릅니다. 전자는 입력을 고치고 후자는 다른 자료나 질문이 필요한지 안내합니다.


### 셀 03 · 공백 질문 전송

Pydantic의 입력 검사를 통과하지 못한 요청의 상태 코드를 확인하세요.

In [3]:
bad = client.post("/ask", json={"question": "   "})
print(bad.status_code)

422


422입니다. 질문을 검사하는 단계에서 멈춰 서비스 함수에는 도달하지 않았습니다.

### 과제 04 · 형식은 맞지만 근거가 없는 질문

주차비를 묻는 POST를 보내 HTTP 코드와 답변의 상태를 출력하세요.

In [4]:
# client.post에 주차 질문을 넣으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>reply = client.post(&#x27;/ask&#x27;, json={&#x27;question&#x27;: &#x27;주차는 무료인가요?&#x27;})
print(reply.status_code, reply.json()[&#x27;result&#x27;][&#x27;status&#x27;])</code></pre>
<p>실행 결과</p><pre><code>200 no_evidence</code></pre>
<p>200 no_evidence입니다. 요청은 정상 처리됐고 제공 자료가 주차비를 다루지 않았습니다.</p>
</details>

## 돌아보기

요청 경로가 틀렸을 때와 JSON 내용이 틀렸을 때 서버가 같은 코드를 줄지 예측해 보세요. 실제 외부 주소·인증·DB 연결은 이 TestClient 검사의 범위 밖입니다.